### Instalação das dependências

Os pacotes abaixo são necessários para o benchmark de modelos com validação cruzada. Execute esta célula antes de qualquer outra.

In [ ]:
!pip install scikit-learn pandas matplotlib -q

# Código 2.1 — Benchmark de Modelos com Validação Cruzada

**Objetivo:** Comparar múltiplos algoritmos de classificação (`RandomForest`, `GradientBoosting` e `LogisticRegression`) em um benchmark com validação cruzada K-Fold estratificada.

### Comparativo de Desempenho dos Algoritmos

| Algoritmo | Complexidade | Interpretabilidade | Indicado para |
| :--- | :--- | :--- | :--- |
| **Logistic Regression** | Baixa | Alta (Pesos diretos) | Linha de base / Modelos simples |
| **Random Forest** | Média | Média (Importância de features) | Dados tabulares não lineares |
| **Gradient Boosting** | Alta | Baixa | Máximo desempenho em dados complexos |


### Passo 1: Configuração do Ambiente e Dados Sintéticos

Verificamos a presença do arquivo de dados sintéticos de churn/falhas.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.datasets import make_classification

if not Path("dados_churn.csv").exists():
    X, y = make_classification(
        n_samples=1000,
        n_features=10,
        n_informative=6,
        random_state=42,
        weights=[0.8, 0.2],
    )
    df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(10)])
    df["churn"] = y
    df.to_csv("dados_churn.csv", index=False)
    print("Arquivo dados_churn.csv gerado com sucesso!")
else:
    print("Arquivo dados_churn.csv encontrado.")

### Passo 2: Importação dos Estimadores e Ferramentas Gráficas

Importamos os modelos, o validador `StratifiedKFold` e o Matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Carga dos dados
df = pd.read_csv("dados_churn.csv")
X = df.drop(columns=["churn"])
y = df["churn"]

### Passo 3: Definição dos Candidate Pipelines

Configuramos um dicionário contendo os 3 candidatos a serem avaliados sob as mesmas condições.

In [ ]:
# Dicionário de candidatos
modelos = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    "Gradient Boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
}

### Passo 4: Execução e Visualização do Boxplot dos Folds

Iteramos pelos modelos executando a validação cruzada e exibimos um gráfico de boxplot para comparar a estabilidade entre os folds.

In [ ]:
# Configurar 5-Fold Estratificado
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultados = {}
for nome, pipeline in modelos.items():
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    resultados[nome] = scores
    print(f"{nome:20s} | Média AUC: {scores.mean():.4f} (± {scores.std():.4f})")

# Visualização da Variabilidade entre Folds (Boxplot)
plt.figure(figsize=(8, 4))
plt.boxplot(resultados.values(), labels=resultados.keys(), patch_artist=True)
plt.ylabel("ROC-AUC Score")
plt.title("Comparativo de Desempenho e Estabilidade dos Modelos (5 Folds)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---

## Conclusão da Aula — Benchmark de Modelos

A validação cruzada estratificada em **K-Fold** nos permite comparar múltiplos algoritmos sob as mesmas condições, fornecendo uma estimativa robusta e sem viés da capacidade de generalização de cada modelo.